### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="in_vehicle_coupon_recommendation",
    dataset_year="2017",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5GS4P",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/in_vehicle_coupon_recommendation/ && wget -P local-data-warehouse/in_vehicle_coupon_recommendation/ https://archive.ics.uci.edu/static/public/603/in+vehicle+coupon+recommendation.zip && unzip local-data-warehouse/in_vehicle_coupon_recommendation/in+vehicle+coupon+recommendation.zip -d local-data-warehouse/in_vehicle_coupon_recommendation/
""",
    # References
    academic_reference_bibtex="""@article{wang2017bayesian,
  title={A bayesian framework for learning rule sets for interpretable classification},
  author={Wang, Tong and Rudin, Cynthia and Doshi-Velez, Finale and Liu, Yimin and Klampfl, Erica and MacNeille, Perry},
  journal={Journal of Machine Learning Research},
  volume={18},
  number={70},
  pages={1--37},
  year={2017}
}
""",
    academic_reference_bibtex_key="wang2017bayesian",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We drop column "toCoupon_GEQ5min" since it has always the same value.
- Anomaly: the data has temporal features but the task is time-invariant.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="AcceptCoupon",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="AcceptCoupon",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/in-vehicle-coupon-recommendation.csv")

df.rename(columns={"Y": "AcceptCoupon"}, inplace=True) # Rename target for clarity

feature_names = [
    'destination', 
    'passanger', 
    'weather', 
    'temperature', 
    'time', 
    'coupon',
    'expiration', 
    'gender', 
    'age', 
    'maritalStatus', 
    'has_children',
    'education', 
    'occupation', 
    'income', 
    'car', 
    'Bar', 
    'CoffeeHouse',
    'CarryAway', 
    'RestaurantLessThan20', 
    'Restaurant20To50',
    'toCoupon_GEQ5min', 
    'toCoupon_GEQ15min', 
    'toCoupon_GEQ25min',
    'direction_same', 
    'direction_opp',
    'AcceptCoupon'
]

cat_features = [
    'destination',
    'passanger',
    'weather',
    'coupon',
    'gender',
    'age',
    'maritalStatus',
    'has_children',
    'education',
    'occupation',
    'income',
    'car',
    'Bar', 
    'CoffeeHouse',
    'CarryAway', 
    'RestaurantLessThan20', 
    'Restaurant20To50',
    'toCoupon_GEQ5min', 
    'toCoupon_GEQ15min', 
    'toCoupon_GEQ25min',
    'direction_same', 
    'direction_opp',
    'AcceptCoupon'
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")
df = df.drop(columns=["toCoupon_GEQ5min"])
df = df.rename(columns={"passanger": "passenger"})
df["gender"] = df["gender"].map({"Female": 0, "Male": 1}).astype("category")
df["has_children"] = df["has_children"].map({1: "Yes", 0: "No"}).astype("category")
df["toCoupon_GEQ15min"] = df["toCoupon_GEQ15min"].map({1: "Yes", 0: "No"}).astype("category")
df["toCoupon_GEQ25min"] = df["toCoupon_GEQ25min"].map({1: "Yes", 0: "No"}).astype("category")
df["direction_same"] = df["direction_same"].map({1: "Yes", 0: "No"}).astype("category")
df["direction_opp"] = df["direction_opp"].map({1: "Yes", 0: "No"}).astype("category")
df["AcceptCoupon"] = df["AcceptCoupon"].map({1: "Yes", 0: "No"}).astype("category")

In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,destination,passenger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,has_children,education,occupation,income,car,Bar,CoffeeHouse,CarryAway,RestaurantLessThan20,Restaurant20To50,toCoupon_GEQ15min,toCoupon_GEQ25min,direction_same,direction_opp,AcceptCoupon
0,Home,Partner,Rainy,55,6PM,Coffee House,1d,0,50plus,Married partner,No,Bachelors degree,Sales & Related,$50000 - $62499,NaN,1~3,4~8,less1,1~3,less1,No,No,Yes,No,Yes
1,No Urgent Place,Friend(s),Sunny,80,6PM,Coffee House,1d,0,50plus,Married partner,Yes,Some college - no degree,Personal Care & Service,$87500 - $99999,NaN,never,gt8,less1,1~3,less1,No,No,No,Yes,Yes
2,Work,Alone,Rainy,55,7AM,Bar,1d,0,31,Married partner,Yes,Graduate degree (Masters or Doctorate),Student,$25000 - $37499,NaN,never,gt8,4~8,1~3,less1,Yes,Yes,No,Yes,No
3,Home,Alone,Snowy,30,6PM,Coffee House,1d,0,31,Married partner,No,Some college - no degree,Computer & Mathematical,$100000 or More,NaN,less1,less1,gt8,4~8,less1,Yes,No,No,Yes,Yes
4,Home,Alone,Sunny,55,6PM,Bar,1d,0,50plus,Married partner,Yes,Associates degree,Legal,$100000 or More,NaN,less1,never,gt8,4~8,1~3,No,No,Yes,No,Yes


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 12,684
Columns: 25
Use sampling: False (sample size: 12,684)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['occupation', 'income', 'age', 'education', 'car', 'coupon', 'time', 'CoffeeHouse', 'maritalStatus', 'Bar']
Rows remaining as candidates after top-10 filter: 7,689 (of 12,684)

#### Duplicate Report
Total duplicate rows: 74 (0.58% of dataset)
Duplicate rows ignoring target: 97 (0.76% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,destination,passenger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,has_children,education,occupation,income,car,Bar,CoffeeHouse,CarryAway,RestaurantLessThan20,Restaurant20To50,toCoupon_GEQ15min,toCoupon_GEQ25min,direction_same,direction_opp,AcceptCoupon
0,Home,Partner,Rainy,55,6PM,Coffee House,1d,0,50plus,Married partner,No,Bachelors degree,Sales & Related,$50000 - $62499,NaN,1~3,4~8,less1,1~3,less1,No,No,Yes,No,Yes
1,No Urgent Place,Friend(s),Sunny,80,6PM,Coffee House,1d,0,50plus,Married partner,Yes,Some college - no degree,Personal Care & Service,$87500 - $99999,NaN,never,gt8,less1,1~3,less1,No,No,No,Yes,Yes
2,Work,Alone,Rainy,55,7AM,Bar,1d,0,31,Married partner,Yes,Graduate degree (Masters or Doctorate),Student,$25000 - $37499,NaN,never,gt8,4~8,1~3,less1,Yes,Yes,No,Yes,No
3,Home,Alone,Snowy,30,6PM,Coffee House,1d,0,31,Married partner,No,Some college - no degree,Computer & Mathematical,$100000 or More,NaN,less1,less1,gt8,4~8,less1,Yes,No,No,Yes,Yes
4,Home,Alone,Sunny,55,6PM,Bar,1d,0,50plus,Married partner,Yes,Associates degree,Legal,$100000 or More,NaN,less1,never,gt8,4~8,1~3,No,No,Yes,No,Yes


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,car,category,12576.0,99.15,5.0,"Mazda5, Scooter and motorcycle, do not drive, Car that is too old to install Onstar :D, crossover"
1,CoffeeHouse,category,217.0,1.71,5.0,"less1, 1~3, never, 4~8, gt8"
2,Restaurant20To50,category,189.0,1.49,5.0,"less1, 1~3, never, 4~8, gt8"
3,CarryAway,category,151.0,1.19,5.0,"1~3, 4~8, less1, gt8, never"
4,RestaurantLessThan20,category,130.0,1.02,5.0,"1~3, 4~8, less1, gt8, never"
5,Bar,category,107.0,0.84,5.0,"never, less1, 1~3, 4~8, gt8"
6,destination,category,0.0,0.00,3.0,"No Urgent Place, Home, Work"
7,passenger,category,0.0,0.00,4.0,"Alone, Friend(s), Partner, Kid(s)"
8,weather,category,0.0,0.00,3.0,"Sunny, Snowy, Rainy"
9,coupon,category,0.0,0.00,5.0,"Coffee House, Restaurant(<20), Carry out & Take away, Bar, Restaurant(20-50)"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
temperature,12684.0,63.301798,19.154486,30.0,80.0


In [8]:
# Categorical Feature Statistics
cat_stats

value  count  \
column               rank                                                    
AcceptCoupon         1                                          Yes   7210   
                     2                                           No   5474   
Bar                  1                                        never   5197   
                     2                                        less1   3482   
                     3                                          1~3   2473   
                     4                                          4~8   1076   
                     5                                          gt8    349   
CarryAway            1                                          1~3   4672   
                     2                                          4~8   4258   
                     3                                        less1   1856   
                     4                                          gt8   1594   
                     5                                        never    153   
CoffeeHouse          1                                        less1   3385   
                     2                                          1~3   3225   
                     3                                        never   2962   
                     4                                          4~8   1784   
                     5                                          gt8   1111   
Restaurant20To50     1                                        less1   6077   
                     2                                          1~3   3290   
                     3                                        never   2136   
                     4                                          4~8    728   
                     5                                          gt8    264   
RestaurantLessThan20 1                                          1~3   5376   
                     2                                          4~8   3580   
                     3                                        less1   2093   
                     4                                          gt8   1285   
                     5                                        never    220   
age                  1                                           21   2653   
                     2                                           26   2559   
                     3                                           31   2039   
                     4                                       50plus   1788   
                     5                                           36   1319   
car                  1                                         <NA>  12576   
                     2                                       Mazda5     22   
                     3                       Scooter and motorcycle     22   
                     4                                 do not drive     22   
                     5     Car that is too old to install Onstar :D     21   
coupon               1                                 Coffee House   3996   
                     2                              Restaurant(<20)   2786   
                     3                        Carry out & Take away   2393   
                     4                                          Bar   2017   
                     5                            Restaurant(20-50)   1492   
destination          1                              No Urgent Place   6283   
                     2                                         Home   3237   
                     3                                         Work   3164   
direction_opp        1                                          Yes   9960   
                     2                                           No   2724   
direction_same       1                                           No   9960   
                     2                                          Yes   2724   
education            1                     Some college - no degree   4351   
      

In [9]:
# Target Distribution
target_df

,count,pct
AcceptCoupon,,
Yes,7210,56.84
No,5474,43.16


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019cd2df-1031-7050-80c6-310e226f08dc
4821f3967efc325ecde7f357e4ef7671eb4dcba711204a2ee9fdae9bd0b9d732
